In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import anndata as ad
import snapatac2 as snap
import matplotlib.pyplot as plt
import seaborn as sns
from plotnine import *
import pyranges as pr
import os
import warnings
import requests
from importlib import reload
from umap import UMAP
from sklearn.decomposition import PCA
from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.cross_decomposition import CCA

from tqdm import tqdm
tqdm.pandas()

import sys
sys.path.append('/data1/normantm/multiome_collab')
from perturbseq import *
import multiome as mo

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica']
plt.rcParams['font.size'] = 12
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42

### S3a assignment

In [ ]:
id_mtx = sc.read_10x_h5('/data1/normantm/eli/T7/202511_RPE1_dg50/251204_aggr/outs/count/filtered_feature_bc_matrix.h5', gex_only=False)
df = pd.read_csv('/data1/normantm/eli/T7/202511_RPE1_dg50/251204_aggr/outs/count/crispr_analysis/protospacer_calls_per_cell.csv')
df = df.set_index("cell_barcode").reindex(id_mtx.obs_names).reset_index().rename(columns={'index':'barcode'}).fillna(0)
df['gemgroup'] = df.barcode.str.split('-').str[1]
df['n_features'] = df.num_features.map(lambda x: '0' if x == 0 else '1' if x == 1 else '2' if x == 2 else '3+')
cellranger_ids = df.groupby('gemgroup').n_features.value_counts(normalize= True).to_frame().reset_index()
cellranger_ids['pct_cells'] = cellranger_ids.proportion * 100
cellranger_plot = cellranger_ids.query("n_features != '0'").copy()
cellranger_plot['n_features'] = pd.Categorical(cellranger_plot.n_features, categories = ['3+', '2', '1'], ordered = True)

In [ ]:
p = (
    ggplot(cellranger_plot)
        + geom_col(aes(x = 'gemgroup', y = 'pct_cells', fill = 'n_features'))
        + theme_light()
        + theme(text=element_text(family="Helvetica"))
        + theme(axis_text_x=element_text(size=12),  # Set X-axis label font size
                axis_text_y=element_text(size=12))
        + theme(figure_size=(3.5, 1.375))
        + theme(legend_position = (0.05,0.9))
        + ylim(0,100)
        + scale_fill_manual(sns.color_palette("rocket", 3).as_hex())
        + labs(x = '', y = 'Percent cells assigned', fill = 'IDs')
)
p.save("plots/S2a_cellranger_assigment.pdf")
p.draw()

In [ ]:
adata = id_mtx[:,id_mtx.var.feature_types == 'CRISPR Guide Capture'].copy()
adata.var_names = adata.var["gene_ids"]
X = adata.X.tocoo()

rows = X.row
cols = X.col
data = X.data

mask = data >= 0

df = pd.DataFrame({
    "barcode": adata.obs_names[rows[mask]].values,
    "feature": adata.var_names[cols[mask]].values,
    "umi_count": data[mask]
})

median_thresholds = dict(df.groupby('feature').umi_count.apply(np.percentile, 25).apply(lambda x: np.maximum(x, 3)))
df['called_median_threshold'] = df.progress_apply(lambda df: df.umi_count >= median_thresholds[df['feature']], axis=1)
df['n_called_median_threshold'] = df.groupby('barcode')['called_median_threshold'].transform('sum')

In [ ]:
df_plot = df[['barcode', 'n_called_median_threshold']].drop_duplicates().set_index('barcode').reindex(id_mtx.obs_names).fillna(0)
df_plot['gemgroup'] = df_plot.index.str.split('-').str[1]
df_plot['n_features'] = df_plot.n_called_median_threshold.map(lambda x: '0' if x == 0 else '1' if x == 1 else '2' if x == 2 else '3+')
threshold_ids = df_plot.groupby('gemgroup').n_features.value_counts(normalize= True).to_frame().reset_index()
threshold_ids['pct_cells'] = threshold_ids.proportion * 100
threshold_plot = threshold_ids.query("n_features != '0'").copy()
threshold_plot['n_features'] = pd.Categorical(threshold_plot.n_features, categories = ['3+', '2', '1'], ordered = True)

In [ ]:
p = (
    ggplot(threshold_plot)
        + geom_col(aes(x = 'gemgroup', y = 'pct_cells', fill = 'n_features'))
        + theme_light()
        + theme(text=element_text(family="Helvetica"))
        + theme(axis_text_x=element_text(size=12),  # Set X-axis label font size
                axis_text_y=element_text(size=12))
        + theme(figure_size=(3.5, 1.375))
        + theme(legend_position = (0.05,0.9))
        + ylim(0,100)
        + scale_fill_manual(sns.color_palette("rocket", 3).as_hex())
        + labs(x = '', y = 'Percent cells assigned', fill = 'IDs')
)
p.save("plots/S3a_threshold_assigment.pdf")
p.draw()

In [ ]:
threshold_plot.query("n_features == '2'").pct_cells.median()

In [ ]:
cellranger_plot.query("n_features == '2'").pct_cells.median()

In [ ]:
full_mtx = sc.read_10x_h5('/data1/normantm/eli/T7/202511_RPE1_dg50/251204_aggr/outs/count/filtered_feature_bc_matrix.h5')
ids_compare = pd.read_csv("/data1/normantm/eli/T7/202511_RPE1_dg50/analysis/intermediate_files/251204_ids_union.csv", index_col='barcode')
assigned = full_mtx[full_mtx.obs_names.isin(ids_compare.index), :].copy()
assigned.obs = assigned.obs.join(ids_compare)
assigned.var_names_make_unique()
assigned.var["mito"] = assigned.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(assigned, inplace=True, qc_vars = ["mito"])
fig, ax = plt.subplots(1, 3, figsize=(9,5), sharey = True)
sns.histplot(data = assigned.obs, x = 'total_counts', kde = True, ax = ax[0], log_scale=True, color = '#0390fc', edgecolor = 'white', rasterized = True)
sns.histplot(data = assigned.obs, x = 'n_genes_by_counts', kde = True, ax = ax[1], log_scale=True, color = '#0390fc', edgecolor = 'white', rasterized = True)
sns.histplot(data = assigned.obs, x = 'pct_counts_mito', kde = True, ax = ax[2], color = '#0390fc', edgecolor = 'white', rasterized = True)
ax[0].set_xlim([1000,20000])
ax[1].set_xlim([500,11000])
ax[0].axvline(x = np.expm1(7.75), linestyle = '--', color = 'red')
ax[1].axvline(x = np.expm1(7), linestyle = '--', color = 'red')
ax[0].set_xlabel("Total UMI")
ax[1].set_xlabel("Unique genes")
ax[2].set_xlabel("UMIs from mitochondrial genes (%)")
plt.tight_layout()
plt.savefig("plots/S3b_dg50_qc.pdf", dpi = 300)
plt.show()

In [ ]:
sc.pp.filter_cells(assigned, min_counts = np.expm1(7.75))
sc.pp.filter_cells(assigned, min_genes = np.expm1(7))
assigned.obs.total_counts.mean(), assigned.obs.n_genes_by_counts.mean()

In [ ]:
gex = sc.read('/data1/normantm/eli/T7/202511_RPE1_dg50/analysis/intermediate_files/251204_dg50_gex_raw.h5ad')
ncells = gex.obs.guide_target.value_counts().to_frame('ncells').reset_index()

In [ ]:
ncells.query("not guide_target.str.contains('_')")

### S3c AB/BA correlation

In [ ]:
gex = sc.read("/data1/normantm/eli/T7/202511_RPE1_dg50/analysis/intermediate_files/251204_dg50_gex.h5ad")
gex.obs['feature_call'] = gex.obs.progress_apply(lambda df: f"{df.target_a.split('_')[0]}_{df.target_b.split('_')[0]}" if not pd.isna(df.target_a) else f"{df.threshold_feature_a.split('_')[0]}_{df.threshold_feature_b.split('_')[0]}", axis=1)
df_pca = pd.DataFrame(gex.obsm['X_pca'], index = gex.obs.feature_call).groupby('feature_call').mean()

def get_correlation(first, second):
    if f"{first}_{second}" in df_pca.index and f"{second}_{first}" in df_pca.index:
        return np.corrcoef(df_pca.loc[f"{first}_{second}"], df_pca.loc[f"{second}_{first}"])[0,1]
    else:
        return np.nan

df_random = pd.DataFrame({"corr": df_pca.T.corr().values.ravel()}).sample(frac = 0.01, random_state = 42)
df_corrs = pd.DataFrame(index = [g for g in gex.obs.guide_target.unique()])
df_corrs.index = df_corrs.index.map(lambda x: f"{x}_NTC" if '_' not in x else x)
df_corrs['first'] = df_corrs.index.map(lambda x: x.split('_')[0])
df_corrs['second'] = df_corrs.index.map(lambda x: x.split('_')[1] if '_' in x else 'NTC')
df_corrs.query("first != 'NTC' and second != 'NTC'")
df_corrs['corr'] = df_corrs.progress_apply(lambda row: get_correlation(row['first'], row['second']), axis=1)

plt.figure(figsize = (3.5,3))
sns.histplot(data = df_random.dropna(), x = 'corr', color = '#bbbbbb', binwidth = 0.05, kde = True, edgecolor = 'white', stat = 'percent', line_kws={'linewidth': 4}, label = 'All genes')
sns.histplot(data = df_corrs.dropna(), x = 'corr', color = '#0390fc', binwidth = 0.05, kde = True, edgecolor = 'white', stat = 'percent', line_kws={'linewidth': 4}, label = 'Same targets')
plt.xlabel('Pearson correlation')
plt.ylabel('% pairs')
plt.legend()
plt.tight_layout()
plt.savefig("S3c_ont_correlation.pdf")
plt.show()

print(df_corrs.dropna()['corr'].median())
print(df_random.dropna()['corr'].median())

### S3d cell number per target

In [ ]:
gex = sc.read('/data1/normantm/eli/T7/202511_RPE1_dg50/analysis/intermediate_files/251204_dg50_gex_raw.h5ad')
ncells = gex.obs.guide_target.value_counts().to_frame('ncells').reset_index()
ncells['identity'] = ncells.guide_target.map(lambda i: 'NTC' if i == 'NTC' else 'Double' if "_" in i else 'Single')
plt.figure(figsize = (30,6))
sns.barplot(ncells, x = 'guide_target', y = 'ncells', hue = 'identity', palette = ['#aaaaaa',  '#00b0f0', '#f77181',], order = ncells.sort_values('ncells', ascending=False).guide_target.tolist(), rasterized = True, legend = None)
plt.yscale('log')
plt.xticks([])
plt.ylabel("Cell number")
plt.xlabel("Perturbed subpopulation")
plt.yticks(fontsize = 16)
plt.tight_layout()
plt.xlabel('Perturbed subpopulation', fontsize=20)
plt.ylabel('Cell number', fontsize=20)
plt.savefig("S3c_ncells.pdf")
plt.show()